In [1]:
import os
import requests
import numpy as np
from time import sleep
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

d:\Github\movie-discovery-app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_API_KEY")
TMDB_URL = "https://api.themoviedb.org/3/"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 774.86it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def get_user_input():
    inputs = []
    print("Enter 2–5 words or phrases (type 'done' to finish):")

    while len(inputs) < 5:
        text = input("> ").strip()
        if text.lower() == "done":
            break
        if text:
            inputs.append(text)

    if len(inputs) < 2:
        raise ValueError("Please enter at least 2 inputs.")

    return inputs

In [4]:
def fetch_movies(pages=3):
    movies = []

    for page in range(1, pages + 1):
        response = requests.get(
            TMDB_URL+"discover/movie",
            params={
                "api_key": TMDB_API_KEY,
                "language": "en-US",
                "sort_by": "vote_count.desc",
                "page": page,
            },
        )
        data = response.json()
        for m in data.get("results", []):
            if m.get("overview"):
                movies.append({
                    "id": m["id"],
                    "title": m["title"],
                    "year": m["release_date"][:4] if m.get("release_date") else "N/A",
                    "overview": m["overview"],
                    "keywords": []
                })

    return movies

# Different Movie queries to be added to the database.

In [5]:
def fetch_keywords(movies):
    for movie in movies:
        response = requests.get(
            TMDB_URL+f"movie/{movie['id']}/keywords",
            params={"api_key": TMDB_API_KEY}
        )
        data = response.json()
        movie["keywords"] = [k["name"] for k in data.get("keywords", [])]
        sleep(0.1)

In [6]:
def movie_keyword_text(movie):
    return ', '.join(movie.get("keywords", []))

def movie_overview_text(movie):
    return movie.get("overview", "")

def movie_text(movie):
    keywords = ", ".join(movie.get("keywords", []))
    return f"{keywords}. {movie['overview']}"

In [7]:
movies = fetch_movies(pages=15)
fetch_keywords(movies)

In [8]:
#movie_texts = [movie_keyword_text(m) for m in movies]
#movie_texts += [movie_overview_text(m) for m in movies]
# movie_texts = [movie_keyword_text(m) + ", " + movie_overview_text(m) for m in movies]
movie_texts = [movie_text(m) for m in movies]

movie_embeddings = model.encode(
    movie_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 10/10 [00:03<00:00,  2.58it/s]


In [10]:
len(movies)

300

# Trying different input queries

In [9]:
user_inputs = get_user_input()
query = ", ".join(user_inputs)

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

Enter 2–5 words or phrases (type 'done' to finish):


In [10]:
similarities = cosine_similarity(
    [query_embedding],
    movie_embeddings
)[0]

top_indices = np.argsort(similarities)[::-1][:5]

In [11]:
for rank, idx in enumerate(top_indices, start=1):
    movie = movies[idx]
    score = similarities[idx]
    print(f"{rank}. {movie['title']} ({movie['year']}) (score: {score:.3f})")
    print(f"   Keywords: {', '.join(movie.get('keywords', []))}")
    print(f"   {movie['overview']}\n")

1. Spider-Man (2002) (score: 0.561)
   Keywords: new york city, adolescence, photographer, loss of loved one, photography, secret identity, hostility, superhero, spider, bad boss, villain, based on comic, teenage boy, teenage love, evil, super villain, taking responsibility
   After being bitten by a genetically altered spider at Oscorp, nerdy but endearing high school student Peter Parker is endowed with amazing powers to become the superhero known as Spider-Man.

2. Spider-Man 2 (2004) (score: 0.527)
   Keywords: new york city, dual identity, love of one's life, secret identity, superhero, pizza boy, based on comic, sequel, romance, doctor, scientist, tentacle, death, super villain, young adult
   Peter Parker is going through a major identity crisis. Burned out from being Spider-Man, he decides to shelve his superhero alter ego, which leaves the city suffering in the wake of carnage left by the evil Doc Ock. In the meantime, Parker still can't act on his feelings for Mary Jane Watso

# Emotion Reccomender

In [34]:
EMOTION_MAP = {
    "mad": "wholesome calming uplifting feel-good movie",
    "sad": "happy lighthearted comedy warm funny movie",
    "short-tempered": "fast-paced energetic short fun movie",
    "anxious": "comforting grounded reassuring gentle movie",
    "well": "serious thoughtful emotional dramatic movie"
}

def get_emotion_input():
    print("\nHow are you feeling?")
    print("Options:", ", ".join(EMOTION_MAP.keys()))
    emotion = input("> ").strip().lower()

    if emotion not in EMOTION_MAP:
        raise ValueError("Unsupported emotion.")

    return emotion

In [38]:
emotion = get_emotion_input()
emotion_text = EMOTION_MAP[emotion]
query_embedding = model.encode(emotion_text, normalize_embeddings=True)

print("\nUse opposite vector? (y/n)")
if input("> ").strip().lower() == "y":
    query_embedding = -query_embedding


How are you feeling?
Options: mad, sad, short-tempered, anxious, well

Use opposite vector? (y/n)
